In [257]:
%reset -f

# S-1 target sections extraction

In [258]:
import re, sys, datetime, traceback
from pathlib import Path
from typing import List, Dict, Optional, Callable, Tuple
import pandas as pd


BASE_DIR = Path.cwd()
S1_ROOT = Path("/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge")
CIK_TXT = BASE_DIR / "CIK_found_done.txt"
OUT_DIR  = BASE_DIR / "S-1_sections_extraction"; OUT_DIR.mkdir(exist_ok=True, parents=True)
CHAPTER_DIR = OUT_DIR / "chapters"; CHAPTER_DIR.mkdir(exist_ok=True, parents=True)

In [259]:
TARGET_SECTIONS = [
    "prospectus summary",
    "risk factors",
    "use of proceeds",
    "management's discussion and analysis",
    "business",
    "management",
]

SYNONYMS = {
    # Summary
    "prospectus summary": [
        "summary",
        "summary of prospectus",
    ],

    # Risk Factors
    "risk factors": [
        "risk factor",
    ],

    # Use of Proceeds
    "use of proceeds": [
        "use of the proceeds",
        "use of offering proceeds",
        "use of net proceeds",
        "application of proceeds",
    ],

    # MD&A
    "management's discussion and analysis": [
        "management's discussion and analysis of financial condition and results of operations",
        "management's discussion and analysis of financial condition and results of operation",
        "managements discussion and analysis of financial condition and results of operations",
        "md&a",
    ],

    # Business
    "business": [
        "our business",
        "business overview",
        "business of the company",
    ],

    # Management
    "management": [
        "directors and executive officers",
        "management and directors",
        "management, directors and executive officers",
        "directors, executive officers and key employees",
        "management and key employees",
    ],

}

### Basic tools

In [260]:
def read_text_safely(path: Path) -> str:
    try:
        txt = path.read_text(encoding="utf-8")
        print(f"[INFO] file {path} is successfully read using utf-8, length {len(txt)}")
        return txt
    except UnicodeDecodeError:
        txt = path.read_text(encoding="latin-1", errors="ignore")
        print(f"[WARN] file {path} is successfully read using latin-1, length {len(txt)}")
        return txt
    
def strip_html(s: str) -> str:
    s = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", s)  # match and remove <script>…</script> and <style>…</style> code blocks
    s = re.sub(r"(?is)<[^>]+>", " ", s)  # remove html tags
    s = re.sub(r"[ \t\u00A0\u3000]+", " ", s)  # collapse multiple spaces, tabs, and non-breaking spaces (\u00A0) into a single regular space
    s = re.sub(r"\n{2,}", "\n", s).strip()
    return s

# simple normalization
def _canon(s: str) -> str:  
    return re.sub(r"\s+", " ", s.lower().replace("’", "'").replace("‘", "'")).strip()

# strict normalization
def canon(s: str) -> str:  
    s = s.lower().replace("’", "'").replace("‘", "'")
    s = re.sub(r"[^a-z0-9'\s&/-]+", " ", s) 
    s = re.sub(r"[\s\u00A0\u2000-\u200B\u202F\u205F\u3000]+", " ", s).strip()
    return s


DEBUG = False

def dbg(*args, flush=True):
    if DEBUG:
        print("[DBG]", *args, flush=flush)

def _build_canon_target_map(TARGET_SECTIONS, SYNONYMS):
    m = {}
    for base in TARGET_SECTIONS:
        m[canon(base)] = base
        for alt in SYNONYMS.get(base, []):
            m[canon(alt)] = base
    return m

### Extract the first TEXT block and TOC

In [261]:

def extract_first_text_block(text: str) -> str:
    m = re.search(r"(?is)<TEXT>(.*?)</TEXT>", text)  # find the first <TEXT>...</TEXT> block
    if not m:
        raise RuntimeError("No <TEXT>...</TEXT> block was found in the filing.")
    return m.group(1)


def extract_first_toc_block(text: str, max_fallback_chars: int = 20000):
    '''locate the standalone TOC title and the first <TABLE> behind it'''
    m = re.search(r'(?im)^\s*table\s+of\s+contents\s*$', text)  # search the standalone "table of contents" and store the first matching object in m
    if not m: return None
    start = m.start()  # record the starting index of the table-of-contents heading in the full text
    first_table = re.search(r'(?is)<TABLE.*?</TABLE>', text[start:])  # search the first TABLE from the starting index
    if first_table:
        end = start + first_table.end() 
        pos = end  # check if there is adjcent table
        while True:  # swallow multiple consecutive directory tables at once
            m_next = re.match(r'(?is)\s*<TABLE.*?</TABLE>', text[pos:])
            if not m_next: break
            end = pos + m_next.end(); pos = end
        return text[start:end]
  
    
def _pick_target_for_toc_title(toc_title: str, canon_map: Dict[str, str]):
    key = _canon(toc_title)
    return (canon_map[key], "exact") if key in canon_map else (None, "")


### Parse TOC and split pages

In [262]:
def parse_toc_entries(toc_block: str) -> List[Dict]:
    toc_text = strip_html(toc_block)
    entries: List[Dict] = []
    pattern = re.compile(r"^(.+?)[\s.\u00A0\u3000\u2026\u00B7]+(\d{1,3})$") 

    noise_rules = [
        re.compile(r'(?i)^(?:table\s+of\s+contents|contents|index|pages?|continued)\s*$'),  # head noises
        re.compile(r'^[\s.\u00A0\u3000\u2026\u00B7\-–—_]+$'),  # pure separating/symbol line
        re.compile(r'(?i)^(?:<\s*/?\s*[a-z0-9]+\s*>)+$'),  # html tags
        re.compile(r'^\s*\d{1,4}\s*$')  # pure number
    ]

    def is_noise(line: str) -> bool:
        return any(rx.match(line) for rx in noise_rules)
    
    buf = ""

    for raw in toc_text.splitlines():  # traverse the table-of-contents text line by line
        line = raw.strip()  # remove leading and trailing whitespace
        if not line:
            continue  # skip blank line
        if is_noise(line):  # skip noises
            continue
        buf = (f"{buf} {line}").strip() if buf else line 
        if not re.search(r"\d{1,3}\s*$", buf):  # full match only when the buffered "logical line" ends with a 1–3 digit number
            continue
        m = pattern.match(buf)
        if not m: continue
        title_raw = m.group(1).strip(); page = int(m.group(2))   # group(1) captures section title; group(2) captures page number
        entries.append({"toc_title": title_raw, "start_page": page})
        buf = ""  # clear buffer

    entries.sort(key=lambda d: d["start_page"])   # sort entries in ascending order by page
    for i, e in enumerate(entries):  # add "end_page" to each entry
        e["end_page"] = entries[i+1]["start_page"] if i+1 < len(entries) else None
    
    '''for debug print'''
    if DEBUG:
        print("[TOC] titles:", [e["toc_title"] for e in entries])
    
    return entries


def split_into_pages(txt: str) -> List[Tuple[Optional[int], str]]:

    txt_norm = re.sub(r"\r\n?", "\n", txt)
    lines = txt_norm.split("\n")

    # capture standalone pure number or -num- or - num -
    def parse_page_marker(s: str) -> Optional[int]:
        s = s.strip()
        m = re.fullmatch(r"(?:([0-9]{1,3})|-\s*([0-9]{1,3})\s*-)", s)
        if not m:
            return None
        return int(m.group(1) or m.group(2))

    # collect the line indecies and number in all page lines
    num_markers: List[Tuple[int, int]] = []
    for idx, ln in enumerate(lines):
        n = parse_page_marker(ln)
        if n is not None:
            num_markers.append((idx, n))

    if not num_markers:
        print("[ERR] No page-number lines detected")
        raise RuntimeError("NO_PAGE_NUMBER_MARKERS")

    pages: List[Tuple[Optional[int], str]] = []

    # the content between two adjacent page number rows belongs to the right page number
    for (i_idx, i_num), (j_idx, j_num) in zip(num_markers, num_markers[1:]):
        seg_lines = lines[i_idx + 1 : j_idx]  # not include page lines
        page_text = "\n".join(seg_lines).strip("\n")
        pages.append((j_num, page_text))

    return pages

### Locate headings

In [263]:
# find the standalone heading
def _find_heading(page_text: str, heading: str) -> Optional[Tuple[int, int]]:
    pat1 = re.compile(rf"(?i)(^|\r?\n)[ \t]*{re.escape(heading)}[ \t]*(\r?\n)")
    m = pat1.search(page_text)
    if m:
        return (m.start(), m.end())

# find the heading, allow linebreaks within the heading
def _find_heading_grow_prefix_and_join_next_line(
    page_text: str,
    heading: str,
    *,
    min_effective_len: int = 1,
    context_window: int = 200,
):
    print(f"[grow] ENTER heading={repr(heading)}", flush=True)

    if not heading:
        print("[grow] ABORT: empty heading", flush=True)
        return None

    target_c = _canon(heading)
    if not target_c:
        print("[grow] ABORT: empty canon target", flush=True)
        return None

    lines = []
    offsets = []  # record the indecies of (start, end)
    pos = 0
    n = len(page_text)
    while pos <= n:
        nl = page_text.find("\n", pos)
        if nl == -1:  # cannot find "\n", meaning from pos to the end is the last line
            start, end = pos, n
            lines.append(page_text[start:end])
            offsets.append((start, end))
            break
        else:
            start, end = pos, nl + 1  
            lines.append(page_text[start:end])
            offsets.append((start, end))
            pos = nl + 1
    
    # match prefix
    best_idx = -1
    best_len = -1
    for i, (raw, (st, ed)) in enumerate(zip(lines, offsets)):
        line_core = raw.rstrip("\r\n")
        line_c = _canon(line_core)
        if not line_c:
            continue
        if len(line_c) < min_effective_len:
            continue
        if target_c.startswith(line_c):
            print(f"[grow] i={i} len={len(line_c)} line_c={repr(line_c)[:200]}", flush=True)
            if len(line_c) > best_len:
                best_len = len(line_c)
                best_idx = i  # update line index
    if best_idx < 0:
        return None
    
    # joint the next line
    st, ed = offsets[best_idx]
    cur_line_core = lines[best_idx].rstrip("\r\n").strip()
    if best_idx + 1 >= len(lines):
        return None
    nst, ned = offsets[best_idx + 1]
    nxt_line_core = lines[best_idx + 1].rstrip("\r\n").strip()
    joined = f"{cur_line_core} {nxt_line_core}"
    joined_c = _canon(joined)

    if joined_c == target_c:
        ctx_end = min(len(page_text), ned + context_window)
        context = page_text[st:ctx_end].replace("\r", "")
        print("[grow] OK", repr(heading), "context=", repr(context), flush=True)
        return (st, ned)

    return None


def _find_heading_two_phase(page_text: str, heading: str, *,
    report=None):
    
    def log(msg: str):
        if report:
            report(msg)
        else:
            print(msg, flush=True)

    span = _find_heading(page_text, heading)
    if span is not None:
        st, ed = span
        snippet = page_text[max(0, st-40): min(len(page_text), ed+40)]
        log(f"[two-phase] phase-1 exact-line matched; skip grow. where={st}-{ed} context={snippet!r}")
        return span
    
    log(f"[two-phase] phase-1 missed; try grow. heading={heading!r}")
    return _find_heading_grow_prefix_and_join_next_line(
        page_text,
        heading,
        min_effective_len=1,
    )

### Extract sections using pages

In [264]:
def extract_section_strict_by_printed_pages(
    pages_with_nums: List[Tuple[Optional[int], str]],
    this_ent: Dict,     # {"toc_title": str, "start_page": int}
    next_ent: Optional[Dict]  # {"toc_title": str, "start_page": int}
) -> str:
    
    printed_to_idx: Dict[int, int] = {}
    for idx, (num, _) in enumerate(pages_with_nums):
        if num is not None and num not in printed_to_idx:
            printed_to_idx[num] = idx

    hdr = this_ent["toc_title"]
    start_printed = int(this_ent["start_page"])
    start_idx = printed_to_idx.get(start_printed, None)
    if start_idx is None:
        return ""  

    start_page_text = pages_with_nums[start_idx][1]

    span = _find_heading_two_phase(start_page_text, hdr)

    if span is None:
        return ""  # if start pages does not contain the standalone target title or title with linebreaks

    _, start_heading_end = span
    parts: List[str] = []
    parts.append(start_page_text[start_heading_end:])

    # extract content until the end of the file if there is no next_ent
    if not next_ent:
        for j in range(start_idx + 1, len(pages_with_nums)):
            parts.append(pages_with_nums[j][1])
        return "\n".join(parts).strip()

    # determine the end title and end page when there is next_ent
    next_hdr = next_ent["toc_title"]
    end_printed = int(next_ent["start_page"])
    end_idx = printed_to_idx.get(end_printed, None)

    # if there is no end page in toc, extract content until the end of the file
    if end_idx is None:
        for j in range(start_idx + 1, len(pages_with_nums)):
            parts.append(pages_with_nums[j][1])
        return "\n".join(parts).strip()

    # add body text after the start page and before the end page
    for j in range(start_idx + 1, end_idx):
        parts.append(pages_with_nums[j][1])

    end_page_text = pages_with_nums[end_idx][1]
    end_span = _find_heading_two_phase(end_page_text, next_hdr)
    

    if end_idx == start_idx:
        if end_span is not None:
            end_heading_start, _ = end_span
            return end_page_text[start_heading_end:end_heading_start].strip()
        else:
            return end_page_text[start_heading_end:].strip()
    
    if end_span is not None:
        end_heading_start, _ = end_span
        parts.append(end_page_text[:end_heading_start])
    else:
        parts.append(end_page_text)

    return "\n".join(parts).strip()


def locate_sections(
    full_txt: str,
    all_toc_entries: List[Dict],
    target_toc_entries: List[Dict],
) -> List[Dict]:

    pages_with_nums: List[Tuple[Optional[int], str]] = split_into_pages(full_txt)

    sections: List[Dict] = []

    def _canon_title(x: str) -> str:
        return re.sub(r"\s+", " ", x or "").strip().lower()

    # contruct indecies for toc
    index_map: Dict[Tuple[int, str], int] = {}
    for i, ent in enumerate(all_toc_entries):
        if "start_page" in ent and "toc_title" in ent:
            try:
                key = (int(ent["start_page"]), _canon_title(ent["toc_title"]))
                if key not in index_map:
                    index_map[key] = i
            except Exception:
                pass

    # extract target sections
    for t in target_toc_entries:
        raw_heading = t.get("toc_title")
        canon_heading = _canon_title(raw_heading)
        raw_start = t.get("start_page")
        print(f"[TARGET] section={t.get('section')}  heading={raw_heading!r}  canon={canon_heading!r}  start_page={raw_start!r}")
        
        try:
            t_key = (int(t["start_page"]), _canon_title(t["toc_title"]))
        except Exception:  # if t is missing or page cannot be int, return ""
            sections.append({"section": t.get("section"), "heading": t.get("toc_title"), "text": ""})
            continue

        idx_in_all = index_map.get(t_key)
        nxt = None
        if idx_in_all is not None and idx_in_all + 1 < len(all_toc_entries):
            nxt = all_toc_entries[idx_in_all + 1]

        # use page to extract
        body = extract_section_strict_by_printed_pages(pages_with_nums, t, nxt)

        sections.append({
            "section": t.get("section"),
            "heading": t.get("toc_title"),
            "text": body
        })

    return sections

### Read CIK

In [265]:
def read_cik_list_from_txt(txt_path: Path) -> List[str]:
    if not txt_path.exists(): raise FileNotFoundError(f"The file in the CIK list does not exist: {txt_path}")
    seen, out = set(), []
    for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        v = line.strip()
        if v and not v.startswith("#") and v not in seen:
            seen.add(v); out.append(v)
    return out

def find_cik_folder(root: Path, cik: str) -> Optional[Path]:
    cand = root / cik
    if cand.is_dir(): return cand
    for p in root.iterdir():
        if p.is_dir() and cik in p.name: return p
    return None

def candidate_filing_files(cik_dir: Path) -> List[Path]:
    exts = ('.txt', '.htm', '.html', '.sgm', '.sgml')
    return [f for f in cik_dir.rglob("*") if f.is_file() and f.suffix.lower() in exts]

def extract_filed_date_from_header(file_path: Path) -> Optional[datetime.date]:
    txt = file_path.read_text(encoding='utf-8', errors='ignore')[:20000]
    m = re.search(r'FILED\s+AS\s+OF\s+DATE\W*\s*(\d{8})', txt, re.IGNORECASE)
    if m:
        s = m.group(1)
        try: return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except: pass
    m2 = re.search(r'ACCEPTANCE-?DATETIME\W*\s*(\d{8})', txt, re.IGNORECASE)
    if m2:
        s = m2.group(1)
        try: return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except: pass
    m3 = re.search(r':\s*(\d{8})', txt[:200])
    if m3:
        s = m3.group(1)
        try: return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except: pass
    return None

### Process S-1

In [266]:
def drop_tables(s: str, placeholder: str = "") -> str:
    if not s:
        return s
    pat_closed = re.compile(r"(?is)<\s*table\b[^>]*>.*?</\s*table\s*>")  # match <table>...</table>
    out = pat_closed.sub(placeholder, s)
    return out

# replace <PAGE>\n with ""
def drop_page_markers(s: str) -> str:
    if not s:
        return s
    pat = re.compile(r"(?im)^[ \t]*<\s*PAGE\s*>[ \t]*(?:\r?\n|$)")
    return pat.sub("", s)


def process_one_cik(cik: str) -> Dict:
    result = {"cik": cik, "status": "not_found", "reason": "", "s1_file": None, "out_dir": None}
    
    folder = find_cik_folder(S1_ROOT, cik)
    if folder is None:
        result.update(status="not_found", reason="CIK folder not found")
        return result

    # choose the earliest released S-1
    candidates = candidate_filing_files(folder)
    if not candidates:
        result.update(status="not_found", reason="no filing-like files (.txt) in folder")
        return result
    cand_with_dates = [(f, extract_filed_date_from_header(f)) for f in candidates]
    cand_with_dates.sort(key=lambda x: (x[1] is None, x[1] if x[1] is not None else datetime.date.max))
    chosen_file, chosen_date = cand_with_dates[0]
    result["s1_file"] = str(chosen_file)
    result["filed_date"] = chosen_date.isoformat() if chosen_date else None

    # obtain the first <TEXT>...</TEXT> block
    raw = read_text_safely(chosen_file)
    text1 = extract_first_text_block(raw)
    text1 = re.sub(r"\r\n?", "\n", text1)
    text1 = drop_page_markers(text1)

    sections: List[Dict] = []

    toc_block = extract_first_toc_block(text1)
    can_try_paged = toc_block is not None
    if not can_try_paged:
        result.update(status="garbled", reason="cannot find paged TOC")
        return result
    
    all_toc_entries = parse_toc_entries(toc_block) if can_try_paged else []
    if not all_toc_entries:
        result.update(status="garbled", reason="paged TOC has no entries")
        return result
    
    # filter out target sections
    if all_toc_entries:
        canon_targets_map = _build_canon_target_map(TARGET_SECTIONS, SYNONYMS)

        target_entries = []
        seen_sections = set()  
        for e in all_toc_entries:
            base, how = _pick_target_for_toc_title(e["toc_title"], canon_targets_map)
            if not base:           
                continue
            if base in seen_sections:
                continue
            seen_sections.add(base)
            target_entries.append({
                "section": base,                 
                "toc_title": e["toc_title"],     
                "start_page": e["start_page"],
                "end_page": e.get("end_page"),
            })


    if not target_entries:
        result.update(status="garbled", reason="no target sections found in the paged TOC")
        return result
    
    try:
        sections = locate_sections(text1, all_toc_entries, target_entries)
    except RuntimeError: 
        sections = []


    if not sections or not any((s.get("text") or "").strip() for s in sections):
        result.update(status="garbled", reason="failed to locate sections by paged TOC")
        return result

    chapter_path = CHAPTER_DIR / f"{cik}.txt"
    with chapter_path.open("w", encoding="utf-8") as cf:
        for i, s in enumerate(sections, 1):
            s["text"] = drop_tables(s.get("text") or "", placeholder="")

            cf.write(f"{'#'*10} {i}. {s['section'].upper()} {'#'*10}\n")
            cf.write((s["text"] or "") + "\n\n")

    result.update(status="ok", chapter_file=str(chapter_path))
    return result

### Main

In [267]:
def main():
    if not CIK_TXT.exists():
        print(f"[ERR] cannot find CIK: {CIK_TXT}"); sys.exit(1)
    
    ciks = read_cik_list_from_txt(CIK_TXT)
    #MAX_CIKS = 20
    #ciks = ciks[:MAX_CIKS]
    print(f"[INFO] read {len(ciks)} CIK for processing")

    processed, garbled = [], []
    for cik in ciks:
        print(f"\n[INFO] process CIK {cik} ...")
        try:
            res = process_one_cik(cik)
        except Exception as e:
            print(f"[ERROR] {cik} error: {e}")
            res = {"cik": cik, "status": "garbled", "reason": f"exception: {e}", "s1_file": None}
        if res["status"] == "ok":
            processed.append(res)
            print(f"[OK] CIK {cik} output: {res.get('out_dir')}, chapter: {res.get('chapter_file')}")
        else:
            garbled.append(res)
            print(f"[WARN] CIK {cik} is labeled as {res['status']}: {res.get('reason')}")
    if processed:
        pd.DataFrame(processed).to_excel(OUT_DIR / "section_extraction_processed_s1.xlsx", index=False)
        print(f"[OK] successfully write CIK list: {OUT_DIR/'section_extraction_processed_s1.xlsx'}")
    if garbled:
        pd.DataFrame(garbled).to_excel(OUT_DIR / "garbled_s1.xlsx", index=False)
        print(f"[OK] failed CIK list: {OUT_DIR/'garbled_s1.xlsx'}")

if __name__ == "__main__":
    main()

[INFO] read 1700 CIK for processing

[INFO] process CIK 0000005588 ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge/0000005588/0000950130-96-002127.txt is successfully read using utf-8, length 850422
[TARGET] section=prospectus summary  heading='Prospectus Summary'  canon='prospectus summary'  start_page=4
[two-phase] phase-1 exact-line matched; skip grow. where=1-52 context=' \n                               PROSPECTUS SUMMARY\n   \n  The following summary is qualified'
[two-phase] phase-1 exact-line matched; skip grow. where=1-48 context=' \n                                 RISK FACTORS\n \n  Prospective investors should conside'
[TARGET] section=risk factors  heading='Risk Factors'  canon='risk factors'  start_page=11
[two-phase] phase-1 exact-line matched; skip grow. where=1-48 context=' \n                                 RISK FACTORS\n \n  Prospective investors should conside'
[two-phase] phase-1 exact-line matched; skip grow. where=1-48 context=" \n       

[OK] CIK 0000005588 output: None, chapter: /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_sections_extraction/chapters/0000005588.txt

[INFO] process CIK 0000018169 ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge/0000018169/0000950123-09-034957.txt is successfully read using utf-8, length 7678699
[WARN] CIK 0000018169 is labeled as garbled: cannot find paged TOC

[INFO] process CIK 0000029806 ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge/0000029806/0000950109-02-003159.txt is successfully read using utf-8, length 2953347
[WARN] CIK 0000029806 is labeled as garbled: cannot find paged TOC

[INFO] process CIK 0000054003 ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge/0000054003/0001193125-04-164965.txt is successfully read using utf-8, length 3056752
[WARN] CIK 0000054003 is labeled as garbled: cannot find paged TOC

[INFO] process CIK 0000075677 ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/